## 1 - imports and setup

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

sys.path.insert(0, os.path.abspath('..'))

import config
from src.evaluation import evaluate_performance
from src.experiment import load_experiment_log, log_run
from src.model import (
    train_catboost_final,
    train_lightgbm_final,
    train_xgboost_final,
)

## 2 - load data and best params

In [ ]:
X_train = pd.read_parquet(config.RESULTS_PATH / "X_train_selected.parquet")
X_val   = pd.read_parquet(config.RESULTS_PATH / "X_val_selected.parquet")
X_test  = pd.read_parquet(config.RESULTS_PATH / "X_test_selected.parquet")
y_train = pd.read_parquet(config.RESULTS_PATH / "y_train.parquet").squeeze()
y_val   = pd.read_parquet(config.RESULTS_PATH / "y_val.parquet").squeeze()
y_test  = pd.read_parquet(config.RESULTS_PATH / "y_test.parquet").squeeze()

with open(config.RESULTS_PATH / "best_params_catboost.json")  as f: cat_params = json.load(f)
with open(config.RESULTS_PATH / "best_params_xgboost.json")   as f: xgb_params = json.load(f)
with open(config.RESULTS_PATH / "best_params_lightgbm.json")  as f: lgb_params = json.load(f)

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")
print(f"y_train fraud rate: {y_train.mean():.4f}")
print("\ncatboost params:", cat_params)
print("xgboost params: ", xgb_params)
print("lightgbm params:", lgb_params)

# string cols were label-encoded and originals dropped in pipeline
cat_features = []

## 3 - train final catboost

In [ ]:
models_dir = config.OUTPUT_PATH / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
cat_path = models_dir / 'catboost_final.pkl'

if cat_path.exists():
    cat_model = joblib.load(cat_path)
    print('loaded catboost_final from disk')
else:
    cat_model = train_catboost_final(
        X_train, y_train, X_val, y_val, cat_features, cat_params
    )
    joblib.dump(cat_model, cat_path)
    print('trained and saved catboost_final')

print(f"best iteration: {cat_model.get_best_iteration()}")

## 4 - train final xgboost

In [ ]:
xgb_path = models_dir / 'xgboost_final.pkl'

if xgb_path.exists():
    xgb_model = joblib.load(xgb_path)
    print('loaded xgboost_final from disk')
else:
    xgb_model = train_xgboost_final(
        X_train, y_train, X_val, y_val, xgb_params
    )
    joblib.dump(xgb_model, xgb_path)
    print('trained and saved xgboost_final')

print(f"best iteration: {xgb_model.best_iteration}")

## 5 - train final lightgbm

In [ ]:
lgb_path = models_dir / 'lightgbm_final.pkl'

if lgb_path.exists():
    lgb_booster = joblib.load(lgb_path)
    print('loaded lightgbm_final from disk')
else:
    lgb_booster = train_lightgbm_final(
        X_train, y_train, X_val, y_val, lgb_params
    )
    joblib.dump(lgb_booster, lgb_path)
    print('trained and saved lightgbm_final')

# native booster -- use booster.predict(), not predict_proba()
print(f"best iteration: {lgb_booster.best_iteration}")

## 6 - ensemble

In [ ]:
# val probabilities for each final model
cat_val_proba = cat_model.predict_proba(X_val)[:, 1]
xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
lgb_val_proba = lgb_booster.predict(X_val)

# val AUC per model -- used as ensemble weights
cat_val_auc = roc_auc_score(y_val, cat_val_proba)
xgb_val_auc = roc_auc_score(y_val, xgb_val_proba)
lgb_val_auc = roc_auc_score(y_val, lgb_val_proba)

raw_weights = np.array([cat_val_auc, xgb_val_auc, lgb_val_auc])
weights = raw_weights / raw_weights.sum()

print(f"catboost  val AUC: {cat_val_auc:.5f}  weight: {weights[0]:.4f}")
print(f"xgboost   val AUC: {xgb_val_auc:.5f}  weight: {weights[1]:.4f}")
print(f"lightgbm  val AUC: {lgb_val_auc:.5f}  weight: {weights[2]:.4f}")

ensemble_val_proba = (
    weights[0] * cat_val_proba
    + weights[1] * xgb_val_proba
    + weights[2] * lgb_val_proba
)
ensemble_val_auc = roc_auc_score(y_val, ensemble_val_proba)
print(f"\nensemble  val AUC: {ensemble_val_auc:.5f}")

## 7 - evaluate all final models on val set

In [ ]:
threshold = 0.5

df_scores = pd.DataFrame({
    'catboost_final':    cat_val_proba,
    'xgboost_final':     xgb_val_proba,
    'lightgbm_final':    lgb_val_proba,
    'ensemble_weighted': ensemble_val_proba,
})

comparison = pd.DataFrame([
    evaluate_performance(df_scores, col, y_val.values, threshold=threshold)
    for col in df_scores.columns
]).sort_values('AUC', ascending=False).reset_index(drop=True)

display(comparison)

In [ ]:
# helper to compute full metric dict from proba array
def _metrics(proba, y, thresh=0.5):
    preds = (proba > thresh).astype(int)
    auc     = roc_auc_score(y, proba)
    pr_auc  = average_precision_score(y, proba)
    prec, rec, f1, _ = precision_recall_fscore_support(y, preds, average='binary')
    return {
        'val_auc':       round(float(auc), 5),
        'val_gini':      round(float(2 * auc - 1), 5),
        'val_pr_auc':    round(float(pr_auc), 5),
        'val_precision': round(float(prec), 5),
        'val_recall':    round(float(rec), 5),
        'val_f1':        round(float(f1), 5),
    }

fixed_meta = {'feature_set': 'selected', 'tuned': True}

log_run('catboost_final',    {**cat_params, **fixed_meta},
        _metrics(cat_val_proba, y_val.values),       config.RESULTS_PATH)
log_run('xgboost_final',     {**xgb_params, **fixed_meta},
        _metrics(xgb_val_proba, y_val.values),       config.RESULTS_PATH)
log_run('lightgbm_final',    {**lgb_params, **fixed_meta},
        _metrics(lgb_val_proba, y_val.values),       config.RESULTS_PATH)
log_run('ensemble_weighted', {'models': 'cat+xgb+lgb', **fixed_meta,
                              'weights': weights.tolist()},
        _metrics(ensemble_val_proba, y_val.values),  config.RESULTS_PATH)

## 8 - full experiment log

In [ ]:
experiment_log = load_experiment_log(config.RESULTS_PATH)
display(experiment_log)

**AUC progression across stages**

| stage | model | val AUC |
|---|---|---|
| baseline | decision_tree max_depth=10 | ~0.806 |
| full features | catboost / xgboost / lightgbm | ~0.906-0.915 |
| tuned 20 trials | catboost / xgboost / lightgbm | see log |
| final selected features | catboost / xgboost / lightgbm | see log |
| ensemble weighted | cat + xgb + lgb | see log |

feature selection typically narrows the feature set without hurting AUC. the ensemble usually matches or slightly exceeds the best individual model. gini (2*AUC-1) is the production metric used in most MRM reviews.

## 9 - save ensemble and individual probabilities

In [ ]:
# test set probabilities
cat_test_proba = cat_model.predict_proba(X_test)[:, 1]
xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]
lgb_test_proba = lgb_booster.predict(X_test)

ensemble_test_proba = (
    weights[0] * cat_test_proba
    + weights[1] * xgb_test_proba
    + weights[2] * lgb_test_proba
)

config.RESULTS_PATH.mkdir(parents=True, exist_ok=True)

pd.Series(ensemble_val_proba,  name='ensemble_proba').to_frame().to_parquet(
    config.RESULTS_PATH / "ensemble_val_proba.parquet")
pd.Series(ensemble_test_proba, name='ensemble_proba').to_frame().to_parquet(
    config.RESULTS_PATH / "ensemble_test_proba.parquet")

pd.Series(cat_val_proba,  name='catboost_proba').to_frame().to_parquet(
    config.RESULTS_PATH / "catboost_final_val_proba.parquet")
pd.Series(xgb_val_proba,  name='xgboost_proba').to_frame().to_parquet(
    config.RESULTS_PATH / "xgboost_final_val_proba.parquet")
pd.Series(lgb_val_proba,  name='lightgbm_proba').to_frame().to_parquet(
    config.RESULTS_PATH / "lightgbm_final_val_proba.parquet")

print("saved ensemble_val_proba.parquet")
print("saved ensemble_test_proba.parquet")
print("saved catboost_final_val_proba.parquet")
print("saved xgboost_final_val_proba.parquet")
print("saved lightgbm_final_val_proba.parquet")